# Linear Systems, Subspaces, QR, and Conditioning
## Deep dive from exact algebra to numerically reliable computation

### Learning goals

By the end of this lab, you should be able to:

- connect rank–nullity to the four fundamental subspaces
- decompose a target into a fitted component and an orthogonal residual
- derive least squares from QR rather than the normal equations
- interpret condition number as worst-case sensitivity
- use the pseudoinverse and regularization deliberately rather than mechanically

Complete Notebook 18 sections 2–5 first. Before running a cell, predict dimensions, ranks, and which quantities should be orthogonal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 4.5), 'axes.grid': True})
rng = np.random.default_rng(7)

## 1. The four fundamental subspaces

For an $m\times n$ matrix of rank $r$:

| Subspace | Ambient space | Dimension |
|---|---|---:|
| column space $\mathcal C(A)$ | $\mathbb R^m$ | $r$ |
| left null space $\mathcal N(A^T)$ | $\mathbb R^m$ | $m-r$ |
| row space $\mathcal C(A^T)$ | $\mathbb R^n$ | $r$ |
| null space $\mathcal N(A)$ | $\mathbb R^n$ | $n-r$ |

The complete SVD exposes orthonormal bases for all four at once. Predict the dimensions below before execution.

In [ ]:
A = np.array([
    [1., 0., 1.,  1.],
    [0., 1., 1., -1.],
    [1., 1., 2.,  0.],  # row 1 + row 2
])
U, singular, Vt = np.linalg.svd(A, full_matrices=True)
rank = np.linalg.matrix_rank(A)

column_basis = U[:, :rank]
left_null_basis = U[:, rank:]
row_basis = Vt[:rank].T
null_basis = Vt[rank:].T

print('shape/rank:', A.shape, rank)
print('subspace basis shapes:', column_basis.shape, left_null_basis.shape,
      row_basis.shape, null_basis.shape)

assert rank == 2
assert column_basis.shape == (3, 2) and left_null_basis.shape == (3, 1)
assert row_basis.shape == (4, 2) and null_basis.shape == (4, 2)
assert np.allclose(A @ null_basis, 0, atol=1e-12)
assert np.allclose(A.T @ left_null_basis, 0, atol=1e-12)
assert np.allclose(column_basis.T @ left_null_basis, 0, atol=1e-12)
assert np.allclose(row_basis.T @ null_basis, 0, atol=1e-12)

## 2. Least squares is an orthogonal decomposition

When $b$ is outside the column space, $A\hat x$ is its projection into that space. The residual $r=b-A\hat x$ lies in the left null space, so $A^Tr=0$. This is the geometric content of the normal equations—not a reason to form $A^TA$ numerically.

In [ ]:
b = np.array([2., -1., 4.])
x_hat, *_ = np.linalg.lstsq(A, b, rcond=None)
fitted = A @ x_hat
residual = b - fitted
projector = column_basis @ column_basis.T

print('one minimum-norm solution:', x_hat)
print('fitted:', fitted, 'residual:', residual)
print('residual coordinates in left null space:', left_null_basis.T @ residual)

assert np.allclose(fitted, projector @ b)
assert np.allclose(A.T @ residual, 0, atol=1e-12)
assert np.allclose(b, fitted + residual)
assert np.allclose(np.linalg.pinv(A) @ b, x_hat)

# Investigation: add any null-space vector to x_hat. What changes?
alternative = x_hat + null_basis @ np.array([2., -1.])
assert np.allclose(A @ alternative, fitted)
assert np.linalg.norm(x_hat) <= np.linalg.norm(alternative) + 1e-12

## 3. QR and reliable least squares

For full-column-rank $X=QR$, minimizing $\|Xc-y\|_2$ becomes minimizing $\|Rc-Q^Ty\|_2$, hence $Rc=Q^Ty$. Forming $X^TX$ squares the condition number and can destroy useful precision.

The Vandermonde example is intentionally difficult: its columns become nearly dependent.

In [ ]:
grid = np.linspace(0, 1, 25)
X = np.vander(grid, N=11, increasing=True)
true_c = np.array([(-1.)**k / (k + 1) for k in range(11)])
y = X @ true_c

Q, R = np.linalg.qr(X, mode='reduced')
c_qr = np.linalg.solve(R, Q.T @ y)
c_normal = np.linalg.solve(X.T @ X, X.T @ y)

qr_error = np.linalg.norm(c_qr - true_c)
normal_error = np.linalg.norm(c_normal - true_c)
print('cond(X):', np.linalg.cond(X))
print('cond(X^T X):', np.linalg.cond(X.T @ X))
print('coefficient error — QR:', qr_error, 'normal equations:', normal_error)

assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-10)
assert np.allclose(Q @ R, X)
assert np.linalg.cond(X.T @ X) > 1e10
assert qr_error < normal_error / 1000
assert np.linalg.norm(X @ c_qr - y) < np.linalg.norm(X @ c_normal - y)

## 4. Conditioning as geometric sensitivity

The singular values are the semi-axis lengths produced when a matrix maps the unit sphere. The 2-norm condition number $\kappa_2(A)=\sigma_{max}/\sigma_{min}$ bounds worst-case relative amplification. A large value does not mean every input is unstable; it identifies a vulnerable direction.

In [ ]:
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, small_sv in zip(axes, [1.0, 0.2, 0.05]):
    M = np.diag([1., small_sv])
    ellipse = M @ circle
    ax.plot(ellipse[0], ellipse[1])
    ax.set_aspect('equal')
    ax.set_title(f'singular values 1, {small_sv}\ncondition {1/small_sv:.0f}')
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
plt.tight_layout(); plt.show()

M = np.diag([1., 0.01])
b0 = np.array([1., 0.01])
b1 = np.array([1., 0.011])
x0, x1 = np.linalg.solve(M, b0), np.linalg.solve(M, b1)
input_change = np.linalg.norm(b1-b0) / np.linalg.norm(b0)
solution_change = np.linalg.norm(x1-x0) / np.linalg.norm(x0)
print('relative input change:', input_change)
print('relative solution change:', solution_change)
assert np.isclose(np.linalg.cond(M), 100)
assert solution_change > 50 * input_change

## 5. Pseudoinverse, truncation, and ridge

Inverting a tiny singular value also amplifies noise. Truncated SVD discards selected directions; ridge replaces $1/\sigma_i$ with $\sigma_i/(\sigma_i^2+\lambda)$. Both trade bias for stability. The correct choice depends on the signal and noise—not only on matrix algebra.

In [ ]:
M = np.diag([1., 0.02])
x_true = np.array([1., 1.])
b_clean = M @ x_true
b_noisy = b_clean + np.array([0., 0.01])

lambdas = np.logspace(-7, 0, 120)
ridge_solutions = np.array([
    np.linalg.solve(M.T @ M + lam*np.eye(2), M.T @ b_noisy)
    for lam in lambdas
])
errors = np.linalg.norm(ridge_solutions - x_true, axis=1)
x_unregularized = np.linalg.solve(M, b_noisy)
best = errors.argmin()

plt.figure()
plt.semilogx(lambdas, errors, label='ridge error')
plt.axhline(np.linalg.norm(x_unregularized-x_true), color='C1', ls='--', label='unregularized')
plt.xlabel('lambda'); plt.ylabel('parameter error'); plt.legend(); plt.show()
print('unregularized:', x_unregularized)
print('best illustrated ridge:', ridge_solutions[best], 'lambda:', lambdas[best])

assert np.linalg.norm(x_unregularized-x_true) > 0.49
assert errors[best] < np.linalg.norm(x_unregularized-x_true)
assert np.all(np.isfinite(ridge_solutions))

## Cumulative representation audit

The synthetic activation matrix below contains two latent directions, one nearly duplicated feature, and noise. Diagnose it as if it came from a model layer:

1. estimate numerical rank and condition number
2. construct an orthonormal basis and verify reconstruction
3. measure the residual after a rank-2 approximation
4. explain whether the third raw feature adds a stable independent direction
5. state what the calculation cannot establish about semantic meaning

In [ ]:
samples = 300
latent = rng.normal(size=(samples, 2))
mixing = np.array([[1., 0.], [0., 1.], [1., 1.0001], [2., -1.]])
activations = latent @ mixing.T + 1e-3*rng.normal(size=(samples, 4))
centered = activations - activations.mean(axis=0)
U_a, s_a, Vt_a = np.linalg.svd(centered, full_matrices=False)
rank2 = (U_a[:, :2] * s_a[:2]) @ Vt_a[:2]
relative_residual = np.linalg.norm(centered-rank2) / np.linalg.norm(centered)
Q_a, R_a = np.linalg.qr(centered[:, :2], mode='reduced')

print('singular values:', s_a)
print('condition number:', s_a[0]/s_a[-1])
print('rank-2 relative residual:', relative_residual)

assert np.allclose(Q_a.T @ Q_a, np.eye(2), atol=1e-12)
assert np.allclose(Q_a @ R_a, centered[:, :2])
assert relative_residual < 0.002
assert s_a[1] / s_a[2] > 100

# Explain before continuing: numerical low rank is evidence about geometry,
# not by itself evidence that either singular direction is a human-interpretable feature.

### Cumulative explanation prompts

- Why does the residual belong to the left null space in least squares?
- Why can two algorithms solve the same exact algebraic problem but behave differently numerically?
- Which singular direction is dangerous in the conditioning example, and why?
- In the activation audit, what additional experiment would be needed to make a causal claim about a direction?

Write answers without consulting the preceding text, then identify one calculation you could reproduce from a blank page.